# MIMIC-IV - High-flow Nasal Oxygen / Optiflow

In this notebook, we extract relevant features for the task of intubation risk predictions for critically ill patients who receive HFNO therapy. Please note, most cells in the notebook require the mimiciv_derived tables that are generated using the scripts provided in the MIMIC github repository, available at: https://github.com/MIT-LCP/mimic-code/tree/main/mimic-iv

The purpose of the extraction pipeline is to synthesise a dataset table in a standardised format, which can subsequently be used in other data processing notebooks. The standardised table format allows for straightforward data processing generalisation to alternative ICU databases, such as AmsterdamUMCdb, eICU, and HiRID.

In [ ]:
# Imports:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from datetime import datetime, timedelta

## 1. Database Connection

In [ ]:
import os
# Database connection is read from the environment so no credentials are committed.
# Set these before running, e.g.:
#   export MIMIC_DB=mimiciv MIMIC_USER=postgres MIMIC_PASSWORD=... MIMIC_HOST=localhost MIMIC_PORT=5432
con = psycopg2.connect(
    database=os.environ.get("MIMIC_DB", "mimiciv"),
    user=os.environ.get("MIMIC_USER", "postgres"),
    password=os.environ["MIMIC_PASSWORD"],
    host=os.environ.get("MIMIC_HOST", "localhost"),
    port=os.environ.get("MIMIC_PORT", "5432"),
)
cursor = con.cursor()
cursor.execute('SET SCHEMA \'public, mimiciv_derived, mimiciv_core, mimiciv_hosp, mimiciv_icu, mimiciv_ed;\''); #set search_path to amsterdamumcdb schema

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

## 2. Identifying HFNO / Optiflow related features

### 2.1) HFNO / Optiflow

In [ ]:
query = """
SELECT ventilation_status, COUNT(*)
FROM mimiciv_derived.ventilation 
GROUP BY ventilation_status
"""
hfno_df = pd.read_sql(query, con)
hfno_df

In [ ]:
query = """
WITH hfno_sessions AS (
    SELECT stay_id
    , starttime
    , endtime
    , (endtime - starttime) AS duration
    , ventilation_status
    FROM mimiciv_derived.ventilation 
    WHERE ventilation_status IN
    (
        'HFNC'
    )
)

SELECT stay_id
, SUM(duration) AS total_duration
, AVG(duration) AS avg_duration
, COUNT(duration) AS nr_hfno_sessions
, MIN(duration) AS shortest_session
, MAX(duration) AS longest_session
FROM hfno_sessions
GROUP BY stay_id
"""
hfno_patients = pd.read_sql(query, con)
hfno_patients

In [ ]:
query = """
SELECT stay_id
, starttime
, endtime
, (endtime - starttime) AS duration
, ventilation_status
FROM mimiciv_derived.ventilation 
WHERE ventilation_status IN
(
    'HFNC',
    'NonInvasiveVent'
)
"""
coh_df = pd.read_sql(query, con)
coh_df

In [ ]:
# This query can be used as a sanity check, to display flow-rate statistics for patients who received HFNO.
# By default, this query is not actually executed, as we execute a modified version at the bottom
# of the notebook that is actually included in the final table.
fr_query = """
WITH hfnc_pop AS (
    SELECT stay_id
    , starttime
    , endtime
    , (endtime - starttime) AS duration
    , ventilation_status
    FROM mimiciv_derived.ventilation 
    WHERE ventilation_status IN
    (
        'HFNC'
    )
),
flow_rates AS (
    SELECT hp.stay_id
    , starttime
    , endtime
    , duration
    , ventilation_status
    , ce.value AS flow_rate
    , ce.valuenum AS flow_rate_num
    FROM hfnc_pop hp
    LEFT JOIN mimiciv_icu.chartevents ce
        ON hp.stay_id = ce.stay_id
        AND ce.charttime >= hp.starttime
        AND ce.charttime <= hp.endtime
    WHERE ce.itemid IN
        (
            224691, -- Flow Rate (L)
            227287, -- additional o2 flow
            223834  -- o2 flow
        )
),
hfno_fr AS (
    SELECT stay_id
    , starttime
    , endtime
    , duration
    , MAX(ventilation_status) AS ventilation_status
    , MIN(flow_rate_num) AS min_flow_rate
    , AVG(flow_rate_num) AS mean_flow_rate
    , MAX(flow_rate_num) AS max_flow_rate
    FROM flow_rates
    GROUP BY stay_id, starttime, endtime, duration
)
SELECT * FROM hfno_fr
ORDER BY stay_id
"""
# fr_df = pd.read_sql(fr_query, con)
# fr_df

In [ ]:
# The following subquery is used to retrieve information about first time receiving HFNC.
# We re-use this timestamp to extract the last measurements of the features later in the notebook.
hfnc_subquery = """
WITH cohort AS (
    SELECT DISTINCT stay_id
    FROM mimiciv_derived.ventilation 
    WHERE ventilation_status IN
    (
        'HFNC',
        'NonInvasiveVent'
    )
),
vent_events AS (
    SELECT vent.stay_id
    , starttime
    , endtime
    , (endtime - starttime) AS duration
    , ventilation_status
    , det.admittime AS admittime
    , (starttime - det.admittime) AS time_to_start
    FROM mimiciv_derived.ventilation AS vent
    JOIN cohort ON cohort.stay_id = vent.stay_id
    LEFT JOIN mimiciv_derived.icustay_detail det ON det.stay_id = vent.stay_id
),
iv_times AS (
    SELECT DISTINCT stay_id
    , MIN(starttime) AS iv_firsttime
    , MIN(endtime) AS iv_firstendtime
    FROM (
        SELECT stay_id, starttime, endtime FROM vent_events
        WHERE ventilation_status = 'InvasiveVent'
    ) iv_sessions
    GROUP BY stay_id
),
sub_vent_events AS (
    SELECT ve.*
    , ivt.iv_firsttime
    , ivt.iv_firstendtime
    FROM vent_events ve
    LEFT JOIN iv_times ivt ON ve.stay_id = ivt.stay_id
    WHERE ventilation_status IN (
        'InvasiveVent',
        'HFNC'
    )
),
vent_events_before_iv AS (
    SELECT *
    FROM sub_vent_events
    WHERE (iv_firsttime IS NULL) OR (starttime < iv_firsttime)
),
last_vent_events AS (
    SELECT stay_id
    , MAX(starttime) AS final_starttime
    , MAX(endtime) AS final_endtime
    FROM vent_events_before_iv
    GROUP BY stay_id
),
last_support AS (
    SELECT vebi.stay_id
    , vebi.ventilation_status AS last_vent_type
    , MAX(final_starttime) AS final_starttime
    , MAX(final_endtime) AS final_endtime
    , (MAX(final_endtime) - MAX(final_starttime)) AS duration
    , MAX(iv_firsttime) AS iv_time
    , MAX(iv_firstendtime) AS iv_endtime
    , (MAX(iv_firsttime) - MAX(final_endtime)) AS time_to_intubation
    , (CASE 
        WHEN MAX(iv_firsttime) IS NULL THEN 0 ELSE 1
        END
      ) AS intubated
    FROM vent_events_before_iv vebi
    LEFT JOIN last_vent_events lve ON lve.stay_id = vebi.stay_id
    WHERE starttime = final_starttime
    GROUP BY vebi.stay_id, vebi.ventilation_status
),
support_table AS (
    SELECT ls.*
        , adm.deathtime
        , (CASE
            WHEN ls.intubated = 0 THEN (adm.deathtime - final_endtime)
            ELSE (adm.deathtime - iv_endtime)
           END) AS time_to_death
        , adm.hospital_expire_flag AS inhosp_mortality
        , det.los_icu AS los_icu
        , adm.admittime
        , adm.dischtime
    FROM last_support ls
    LEFT JOIN mimiciv_derived.icustay_detail det
        ON ls.stay_id = det.stay_id
    LEFT JOIN mimiciv_hosp.admissions adm
        ON det.subject_id = adm.subject_id
        AND det.hadm_id = adm.hadm_id
    WHERE duration > '2:00:00' -- Exclude patients with HFNO duration < 2 hours.
        AND (time_to_intubation IS NULL OR time_to_intubation <= '4:00:00') -- Exclude patients with too wide a time gap between HFNO and IV.
        AND (det.los_icu >= 1.0) -- Exclude patients with icu stay < 24 hours.
    ORDER BY stay_id
)
"""

In [ ]:
%%time
query = f"""

{hfnc_subquery}

SELECT * FROM support_table
ORDER BY stay_id

"""
hfno_df = pd.read_sql(query, con)
hfno_df

In [ ]:
import re

# STROBE flowchart: get pre-criteria pool count and per-criterion exclusions
# Strip the WHERE inclusion criteria from support_table to count the full pre-criteria pool
hfnc_subquery_raw = re.sub(
    r"WHERE duration > '2:00:00'.*?ORDER BY stay_id",
    "ORDER BY stay_id",
    hfnc_subquery,
    flags=re.DOTALL
)

flowchart_query = f"""
{hfnc_subquery_raw}
SELECT
    COUNT(*)                                                                                AS total_hfno_before_criteria,
    SUM(CASE WHEN duration <= '2:00:00' THEN 1 ELSE 0 END)                                 AS excl_duration_lt2h,
    SUM(CASE WHEN time_to_intubation > '4:00:00' THEN 1 ELSE 0 END)                        AS excl_gap_gt4h,
    SUM(CASE WHEN los_icu < 1.0 THEN 1 ELSE 0 END)                                         AS excl_los_lt1d,
    SUM(CASE WHEN duration > '2:00:00'
             AND (time_to_intubation IS NULL OR time_to_intubation <= '4:00:00')
             AND los_icu >= 1.0 THEN 1 ELSE 0 END)                                          AS after_criteria
FROM support_table
"""

fc = pd.read_sql(flowchart_query, con)

total  = int(fc['total_hfno_before_criteria'].iloc[0])
excl_d = int(fc['excl_duration_lt2h'].iloc[0])
excl_g = int(fc['excl_gap_gt4h'].iloc[0])
excl_l = int(fc['excl_los_lt1d'].iloc[0])
after  = int(fc['after_criteria'].iloc[0])

print('=== STROBE FLOWCHART — SCREENING COUNTS ===')
print(f'HFNO before intubation (pre-criteria pool) : N = {total}')
print(f'  Excluded - HFNO duration <2 hours         : N = {excl_d}')
print(f'  Excluded - Time gap >4h to intubation     : N = {excl_g}')
print(f'  Excluded - ICU stay <24 hours             : N = {excl_l}')
print(f'  Total unique criteria exclusions (overlap possible): N = {total - after}')
print(f'After inclusion criteria (before DNI/CMO)  : N = {after}')
print(f'  (should match hfno_df rows = {len(hfno_df)})')


#### Exclude patients with DNI or Comfort Measures Only (CMO) status code

In [ ]:
%%time
status_code_query = f"""

{hfnc_subquery}

SELECT ce.stay_id
    , (CASE WHEN value LIKE '%DNI%' THEN 1 ELSE 0 END) AS sc_DNI
    , (CASE WHEN value LIKE '%Comfort measures only%' THEN 1 ELSE 0 END) AS sc_CMO
FROM mimiciv_icu.chartevents ce
LEFT JOIN support_table st
    ON ce.stay_id = st.stay_id
WHERE ce.stay_id IN {tuple(eligible_stayids)}
    AND itemid IN (
        223758,
        228687
    )
    AND ce.charttime <= st.final_endtime
"""
status_code_df = pd.read_sql(status_code_query, con)
status_code_df

In [ ]:
exclude_stayids = status_code_df[(status_code_df.sc_dni == 1) | (status_code_df.sc_cmo == 1)].stay_id.unique()
print(f"Number of patients excluded due to status code: {len(exclude_stayids)}")
eligible_stayids = [id for id in eligible_stayids if not id in exclude_stayids]
print(f"Number of patients remaining: {len(eligible_stayids)}")
hfno_df_orig = hfno_df
hfno_df = hfno_df[hfno_df.stay_id.isin(eligible_stayids)]
hfno_df

#### Add mortality flags

In [ ]:
# We define mortality if the patient dies within a 4 hour window after HFNO treatment was discontinued.
# Although HFNO failure is defined as the need for intubation OR death after HFNO discontinuation, we
# compute the time_to_death / died column for intubated patients based on the IV endtime, not the HFNO endtime.
# This means that the time_to_death delta is computed based on the endtime of the latest therapy provided to the patient.

hfno_df['died'] = (hfno_df['time_to_death'] <= timedelta(hours=4)).astype(int)

hfno_df

## 3. Retrieve features

### 3.1 Demographics

In [ ]:
%%time
demo_query = f"""
SELECT det.stay_id
, admission_age
, gender
, race
, weight_admit
, weight_admit / ((height / 100) * (height/100)) AS bmi
, apsiii
, gcs_min
, dialysis_active
FROM mimiciv_derived.icustay_detail det
LEFT JOIN mimiciv_derived.first_day_height fdh ON fdh.stay_id = det.stay_id
LEFT JOIN mimiciv_derived.first_day_weight fdw ON fdw.stay_id = det.stay_id
LEFT JOIN mimiciv_derived.apsiii ON apsiii.stay_id = det.stay_id
LEFT JOIN mimiciv_derived.first_day_gcs fdg ON fdg.stay_id = det.stay_id
LEFT JOIN mimiciv_derived.first_day_rrt fdr ON fdr.stay_id = det.stay_id
WHERE det.stay_id IN {tuple(eligible_stayids)}
"""
demo_df = pd.read_sql(demo_query, con)
demo_df

### 3.2 Comorbidities

In [ ]:
%%time
# Adapted from: https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iv/concepts/sepsis/sepsis3.sql
sofa_query = f"""
WITH sofa AS (
    SELECT stay_id
    , starttime
    , endtime
    , respiration_24hours AS respiration
    , coagulation_24hours AS coagulation
    , liver_24hours AS liver
    , cardiovascular_24hours AS cardiovascular
    , cns_24hours AS cns
    , renal_24hours AS renal
    , sofa_24hours AS sofa_score
    FROM mimiciv_derived.sofa
    WHERE stay_id IN {tuple(eligible_stayids)}
    ORDER BY stay_id
)

, s1 AS (
    SELECT soi.stay_id
    , soi.suspected_infection
    , soi.suspected_infection_time
    , soi.antibiotic_time
    , soi.culture_time
    -- SOFA COLUMNS
    , starttime
    , endtime
    , respiration, coagulation, liver, cardiovascular, cns, renal
    , sofa_score
    , sofa_score >= 2 AND suspected_infection = 1 AS sepsis3
    , ROW_NUMBER() OVER
    (
        PARTITION BY soi.stay_id
        ORDER BY
            suspected_infection_time, antibiotic_time, culture_time, endtime
    ) AS rn_sus
    FROM mimiciv_derived.suspicion_of_infection AS soi
    INNER JOIN sofa
        ON soi.stay_id = sofa.stay_id
            AND sofa.endtime >= soi.suspected_infection_time + '-48:00:00'
            AND sofa.endtime <= soi.suspected_infection_time + '24:00:00'
    WHERE soi.stay_id IS NOT NULL
        AND soi.stay_id IN {tuple(eligible_stayids)}
    ORDER BY stay_id
)

SELECT stay_id
, antibiotic_time
, culture_time
, suspected_infection_time
, endtime AS sofa_time
, sofa_score
, respiration, coagulation, liver, cardiovascular, cns, renal
, sepsis3
FROM s1
WHERE rn_sus = 1
ORDER BY stay_id
"""
sofa_df = pd.read_sql(sofa_query, con)
sofa_df

# no_data = [x for x in eligible_stayids if x not in sofa_df.stay_id.values.tolist()]
# hfno_df[hfno_df.stay_id.isin(no_data)]

In [ ]:
%%time
comorb_query = f"""
WITH comorb_notes AS (
    SELECT det.stay_id
    -- Sepsis3 score and individual (sofa) components:
    , (CASE
        WHEN sepsis3 = true THEN 1
        ELSE 0
       END) AS sepsis3
    , s3.antibiotic_time AS s3_antibiotic_time
    , s3.sofa_time AS s3_sofa_time
    , s3.sofa_score AS s3_sofa_score
    , s3.respiration AS s3_respiration
    , s3.coagulation AS s3_coagulation
    , s3.liver AS s3_liver
    , s3.cardiovascular AS s3_cardiovascular
    , s3.cns AS s3_cns
    , s3.renal AS s3_renal

    -- Include SOFA measurements separately to use for patients who did not develop sepsis.
    -- In this case, we will use the maximum SOFA scores within 24 hours after admission.
    , sofa.respiration AS sofa_respiration
    , sofa.coagulation AS sofa_coagulation
    , sofa.liver AS sofa_liver
    , sofa.cardiovascular AS sofa_cardiovascular
    , sofa.cns AS sofa_cns
    , sofa.renal AS sofa_renal

    -- Other comorbidities:
    , myocardial_infarct
    , congestive_heart_failure
    , peripheral_vascular_disease
    , cerebrovascular_disease
    , chronic_pulmonary_disease
    , (CASE
        WHEN mild_liver_disease = 1 THEN 1
        WHEN severe_liver_disease = 1 THEN 1
        ELSE 0
       END) AS liver_disease
    , renal_disease
    , malignant_cancer
    , (CASE
        WHEN diabetes_without_cc = 1 THEN 1
        WHEN diabetes_with_cc = 1 THEN 1
        ELSE 0
       END) AS diabetes
    FROM mimiciv_derived.icustay_detail det
    LEFT JOIN mimiciv_derived.sepsis3 s3 ON s3.stay_id = det.stay_id
    LEFT JOIN mimiciv_derived.sofa sofa ON sofa.stay_id = det.stay_id
        AND sofa.endtime >= det.admittime
        AND sofa.endtime <= det.admittime + '24:00:00'
    LEFT JOIN mimiciv_derived.charlson charlson ON charlson.subject_id = det.subject_id
    WHERE det.stay_id IN {tuple(eligible_stayids)}
)

SELECT stay_id
, MAX(sepsis3) AS sepsis3
, MAX(s3_antibiotic_time) AS s3_antibiotic_time
, MAX(s3_sofa_time) AS s3_sofa_time
, MAX(s3_sofa_score) AS s3_sofa_score
, MAX(s3_respiration) AS s3_respiration
, MAX(s3_coagulation) AS s3_coagulation
, MAX(s3_liver) AS s3_liver
, MAX(s3_cardiovascular) AS s3_cardiovascular
, MAX(s3_cns) AS s3_cns
, MAX(s3_renal) AS s3_renal

-- For sepsis patients we use the scores from the sepsis3 table. For non-sepsis patients we try to use SOFA scores from
-- a 24 hour window after admission. If both scores are unavailable, we impute 0.
, COALESCE(MAX(s3_respiration), MAX(sofa_respiration), 0) AS sofa_respiration
, COALESCE(MAX(s3_coagulation), MAX(sofa_coagulation), 0) AS sofa_coagulation
, COALESCE(MAX(s3_liver), MAX(sofa_liver), 0) AS sofa_liver
, COALESCE(MAX(s3_cardiovascular), MAX(sofa_cardiovascular), 0) AS sofa_cardiovascular
, COALESCE(MAX(s3_cns), MAX(sofa_cns), 0) AS sofa_cns
, COALESCE(MAX(s3_renal), MAX(sofa_renal), 0) AS sofa_renal

, MAX(myocardial_infarct) AS myocardial_infarct
, MAX(congestive_heart_failure) AS congestive_heart_failure
, MAX(peripheral_vascular_disease) AS peripheral_vascular_disease
, MAX(cerebrovascular_disease) AS cerebrovascular_disease
, MAX(chronic_pulmonary_disease) AS chronic_pulmonary_disease
, MAX(liver_disease) AS liver_disease
, MAX(renal_disease) AS renal_disease
, MAX(malignant_cancer) AS malignant_cancer
, MAX(diabetes) AS diabetes
FROM comorb_notes
GROUP BY stay_id
ORDER BY stay_id
"""
comorb_df = pd.read_sql(comorb_query, con)
comorb_df


### 3.3 Vital Signs

In [ ]:
%%time
vital_query = f"""
SELECT det.stay_id
, heart_rate_min, heart_rate_max, heart_rate_mean
, sbp_min, sbp_max, sbp_mean
, dbp_min, dbp_max, dbp_mean
, mbp_min, mbp_max, mbp_mean
, resp_rate_min, resp_rate_max, resp_rate_mean
, temperature_min, temperature_max, temperature_mean
, spo2_min, spo2_max, spo2_mean
, glucose_min, glucose_max, glucose_mean
, urineoutput
FROM mimiciv_derived.icustay_detail det
LEFT JOIN mimiciv_derived.first_day_vitalsign fdv ON fdv.stay_id = det.stay_id
LEFT JOIN mimiciv_derived.first_day_urine_output fduo ON fduo.stay_id = det.stay_id
WHERE det.stay_id IN {tuple(eligible_stayids)}
"""
vital_df = pd.read_sql(vital_query, con)
vital_df

In [ ]:
# Create separate vital_df with only the mean values
mean_vital_df = vital_df[['stay_id', 'heart_rate_mean', 'sbp_mean', 'dbp_mean', 'mbp_mean', 'resp_rate_mean', 'temperature_mean', 'spo2_mean', 'glucose_mean', 'urineoutput']]
mean_vital_df

#### Derive fluid balance

In [ ]:
%%time
# Retrieve urine output for last 24h before HFNC as a separate table:
uo_last24h_query = f"""
{hfnc_subquery}

SELECT stay_id, urineoutput_24hr, uo_tm_24hr
FROM (
    SELECT det.stay_id
        , uor.urineoutput_24hr
        , uor.uo_tm_24hr
        , ROW_NUMBER() OVER (PARTITION BY det.stay_id ORDER BY uor.charttime DESC) AS n
    FROM mimiciv_derived.icustay_detail det
    LEFT JOIN support_table st
        ON det.stay_id = st.stay_id
    LEFT JOIN mimiciv_derived.urine_output_rate uor
        ON det.stay_id = uor.stay_id
            AND uor.charttime >= st.final_starttime + '-10:00:00'
            AND uor.charttime <= st.final_starttime + '2:00:00'
    WHERE det.stay_id IN {tuple(eligible_stayids)}
    ORDER BY det.stay_id, uor.charttime DESC
) AS temp
WHERE n = 1
"""
uo_last24h_df = pd.read_sql(uo_last24h_query, con)
uo_last24h_df

In [ ]:
# Fluid input
fluid_last24h_query= f"""
{hfnc_subquery}

, windowed_input AS (
  SELECT
      ie.stay_id,
      ie.starttime,
      ie.endtime,
      ie.rate,              -- usually mL/hour
      ie.rateuom,
      ie.amount,            -- usually total mL
      ie.amountuom,
      ie.itemid,
      st.final_starttime,
      (st.final_starttime - INTERVAL '24 hours') AS window_start
  FROM mimiciv_icu.inputevents ie
  JOIN support_table st
    ON st.stay_id = ie.stay_id
  WHERE ie.stay_id IN {tuple(eligible_stayids)}                                
    -- Keep any event that overlaps the window:
    AND ie.starttime < st.final_starttime + INTERVAL '2 hours'
    AND COALESCE(ie.endtime, st.final_starttime) > st.final_starttime - INTERVAL '22 hours'
    -- Basic unit sanity (adjust if you harmonize units elsewhere):
    AND LOWER(COALESCE(ie.amountuom, '')) = 'ml'
)
, overlapping_frames AS (
  SELECT
      stay_id,
      itemid,
      final_starttime,
      -- clamp start/end to window
      starttime,
      endtime,
      GREATEST(starttime, window_start) AS eff_start,
      LEAST(COALESCE(endtime, final_starttime), final_starttime) AS eff_end,
      rate,
      rateuom,
      amount,
      amountuom
  FROM windowed_input
)
, fluid_durations AS (
    SELECT
      *,
      EXTRACT(EPOCH FROM (eff_end - eff_start))/3600.0 AS overlapped_hours
  FROM overlapping_frames
  WHERE eff_end > eff_start  -- discard zero/negative intersections
)
, sub_amounts AS (
    SELECT stay_id, itemid,
        -- If rate known: use rate * hours but do not exceed known total amount (if present).
        CASE
            WHEN rate IS NOT NULL THEN
                CASE
                    WHEN amount IS NOT NULL THEN LEAST(amount, rate * overlapped_hours)
                    ELSE rate * overlapped_hours
                END
            -- Bolus (no rate): if it overlaps, count the whole amount (already ensured by eff_end>eff_start)
            ELSE COALESCE(amount, 0)
        END AS sub_amount_ml
    FROM fluid_durations
)

SELECT * from sub_amounts
"""

sub_fluid_amounts_df = pd.read_sql(fluid_last24h_query, con)
fluid_last24h_df = (
    sub_fluid_amounts_df
    .groupby("stay_id", as_index=False)['sub_amount_ml']
    .sum()
    .rename(columns={'sub_amount_ml': 'fluidinput_last24h'})
)
fluid_last24h_df

In [ ]:
fluidbalance_df = uo_last24h_df.merge(fluid_last24h_df, on='stay_id', how='left')
fluidbalance_df['fluidinput_last24h'] = fluidbalance_df['fluidinput_last24h'].fillna(0)
fluidbalance_df['fluidbalance_24hr'] = fluidbalance_df['fluidinput_last24h'] - fluidbalance_df['urineoutput_24hr']
fluidbalance_df

#### Derive vasopressor usage

In [ ]:
%%time

vaso_query = f"""
{hfnc_subquery}

, vasopressors AS (
    SELECT
        ie.stay_id,
        ie.itemid,
        ie.starttime,
        ie.endtime,
        st.final_starttime,
        st.final_starttime - INTERVAL '24 hours' AS win24_start,
        st.final_starttime - INTERVAL '12 hours' AS win12_start,
        st.final_starttime - INTERVAL '6 hours'  AS win6_start
    FROM mimiciv_icu.inputevents ie
    JOIN support_table st
        ON ie.stay_id = st.stay_id
    WHERE
        ie.stay_id IN {tuple(eligible_stayids)}  
        AND ie.itemid IN (
          221906, -- Norepinephrine
          221289, -- Epinephrine
          221662, -- Dopamine
          221749, -- Phenylephrine
          222315, -- Vasopressin
          229709, 229764 -- Angiotensin II
        )
)
, vaso_flags AS (
    SELECT
        stay_id,
        -- Clamp endtime to final_starttime if NULL
        GREATEST(COALESCE(endtime, final_starttime), starttime) AS eff_end,
        starttime,
        win24_start,
        win12_start,
        win6_start,
        final_starttime
    FROM vasopressors
)
, has_vaso_last24h AS (
    SELECT DISTINCT stay_id
    FROM vaso_flags
    WHERE starttime < final_starttime AND eff_end > win24_start
)
, has_vaso_last12h AS (
    SELECT DISTINCT stay_id
    FROM vaso_flags
    WHERE starttime < final_starttime AND eff_end > win12_start
)
, has_vaso_last6h AS (
    SELECT DISTINCT stay_id
    FROM vaso_flags
    WHERE starttime < final_starttime AND eff_end > win6_start
)

SELECT st.stay_id
    , CASE WHEN v24.stay_id IS NOT NULL THEN 1 ELSE 0 END AS vp_last24h
    , CASE WHEN v12.stay_id IS NOT NULL THEN 1 ELSE 0 END AS vp_last12h
    , CASE WHEN v6.stay_id IS NOT NULL THEN 1 ELSE 0 END AS vp_last6h
FROM support_table st
LEFT JOIN has_vaso_last24h v24 ON st.stay_id = v24.stay_id
LEFT JOIN has_vaso_last12h v12 ON st.stay_id = v12.stay_id
LEFT JOIN has_vaso_last6h  v6  ON st.stay_id = v6.stay_id
WHERE st.stay_id IN {tuple(eligible_stayids)}  

"""
vaso_df = pd.read_sql(vaso_query, con)
vaso_df

In [ ]:
# %%time
# vs_last_query = f"""
# {hfnc_subquery}

# SELECT DISTINCT det.stay_id
#     , LAST_VALUE(heart_rate) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN heart_rate IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS heart_rate_last
#     , LAST_VALUE(sbp) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN sbp IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS sbp_last
#     , LAST_VALUE(dbp) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN dbp IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS dbp_last
#     , LAST_VALUE(mbp) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN mbp IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS mbp_last
#     , LAST_VALUE(resp_rate) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN resp_rate IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS resp_rate_last
#     , LAST_VALUE(temperature) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN temperature IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS temperature_last
#     , LAST_VALUE(spo2) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN spo2 IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS spo2_last
#     , LAST_VALUE(glucose) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN glucose IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS glucose_vital_last
# FROM mimiciv_derived.icustay_detail det
# LEFT JOIN support_table st
#     ON det.stay_id = st.stay_id
# LEFT JOIN mimiciv_derived.vitalsign vs
#     ON det.stay_id = vs.stay_id
#         AND vs.charttime >= st.final_starttime + '-24:00:00'
#         AND vs.charttime <= st.final_starttime
# WHERE det.stay_id IN {tuple(eligible_stayids)}
# ORDER BY det.stay_id
# """
# vs_last_df = pd.read_sql(vs_last_query, con)
# vs_last_df

#### Alternative: use average vital sign measurements over the last 24 hours

In [ ]:
%%time
vs_mean_last24h_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , AVG(heart_rate) AS heart_rate_mean_last24h
    , AVG(sbp) AS sbp_mean_last24h
    , AVG(dbp) AS dbp_mean_last24h
    , AVG(mbp) AS mbp_mean_last24h
    , AVG(resp_rate) AS resp_rate_mean_last24h
    , AVG(temperature) AS temperature_mean_last24h
    , AVG(spo2) AS spo2_mean_last24h
    , AVG(glucose) AS glucose_mean_last24h
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.vitalsign vs
    ON det.stay_id = vs.stay_id
        AND vs.charttime >= st.final_starttime + '-22:00:00'
        AND vs.charttime <= st.final_starttime + '2:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
GROUP BY det.stay_id
ORDER BY det.stay_id
"""
vs_mean_last24h_df = pd.read_sql(vs_mean_last24h_query, con)
vs_mean_last24h_df

In [ ]:
# Sanity check: Print all measurements between admission (-6 hours) and receiving HFNC.
# temp_q = f"""
# {hfnc_subquery}

# SELECT det.stay_id
#     , vs.charttime
#     , (vs.charttime - det.admittime) AS time_since_adm
#     , ht.hfnc_firsttime
#     , heart_rate
#     , sbp
#     , dbp
#     , mbp
#     , resp_rate
#     , temperature
#     , spo2
#     , glucose
#     , urineoutput
# FROM mimiciv_derived.icustay_detail det
# LEFT JOIN hfnc_times ht
#     ON det.stay_id = ht.stay_id
# LEFT JOIN mimiciv_derived.vitalsign vs
#     ON det.stay_id = vs.stay_id
#         AND vs.charttime >= ht.hfnc_firsttime + '-24:00:00'
#         AND vs.charttime <= ht.hfnc_firsttime
# LEFT JOIN mimiciv_derived.urine_output uo
#     ON det.stay_id = uo.stay_id
#         AND uo.charttime >= ht.hfnc_firsttime + '-24:00:00'
#         AND uo.charttime <= ht.hfnc_firsttime
# WHERE det.stay_id IN {tuple(eligible_stayids)} AND det.stay_id = 30036116
# ORDER BY det.stay_id, charttime
# """
# temp = pd.read_sql(temp_q, con)
# temp.head(500)

### 3.4 Lab values

In [ ]:
%%time
lab_query = f"""
SELECT det.stay_id
, hematocrit_min, hematocrit_max
, hemoglobin_min, hemoglobin_max
, platelets_min, platelets_max
, wbc_min, wbc_max
, albumin_min, albumin_max
, aniongap_min, aniongap_max
, bicarbonate_min, bicarbonate_max
, bun_min, bun_max
, calcium_min, calcium_max
, chloride_min, chloride_max
, creatinine_min, creatinine_max
, sodium_min, sodium_max
, potassium_min, potassium_max
, abs_basophils_min, abs_basophils_max
, abs_eosinophils_min, abs_eosinophils_max
, abs_lymphocytes_min, abs_lymphocytes_max
, abs_monocytes_min, abs_monocytes_max
, abs_neutrophils_min, abs_neutrophils_max
, bands_min, bands_max
, inr_min, inr_max
, ptt_min, ptt_max
, alt_min, alt_max
, bilirubin_total_min, bilirubin_total_max
FROM mimiciv_derived.icustay_detail det
LEFT JOIN mimiciv_derived.first_day_lab fdl ON fdl.stay_id = det.stay_id
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
lab_df = pd.read_sql(lab_query, con)
lab_df

#### Alternative: use the first measurements for the features after ICU admission

In [ ]:
# %%time
# lab_first_query = f"""
# SELECT DISTINCT det.stay_id
# -- From table: complete_blood_count
#     , FIRST_VALUE(cbc.wbc) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.wbc IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS wbc_first
#     , FIRST_VALUE(cbc.hematocrit) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.hematocrit IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS hematocrit_first
#     , FIRST_VALUE(cbc.hemoglobin) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.hemoglobin IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS hemoglobin_first
#     , FIRST_VALUE(cbc.platelet) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.platelet IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS platelet_first
#     , FIRST_VALUE(cbc.mch) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.mch IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS mch_first
#     , FIRST_VALUE(cbc.mchc) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.mchc IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS mchc_first
#     , FIRST_VALUE(cbc.mcv) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.mcv IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS mcv_first
#     , FIRST_VALUE(cbc.rbc) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.rbc IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS rbc_first
#     , FIRST_VALUE(cbc.rdw) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN cbc.rdw IS NOT NULL THEN 0 ELSE 1 END ASC, cbc.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS rdw_first

# -- From table: chemistry
#     , FIRST_VALUE(chem.aniongap) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.aniongap IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS aniongap_first
#     , FIRST_VALUE(chem.bicarbonate) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.bicarbonate IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS bicarbonate_first
#     , FIRST_VALUE(chem.bun) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.bun IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS bun_first
#     , FIRST_VALUE(chem.calcium) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.calcium IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS calcium_first
#     , FIRST_VALUE(chem.chloride) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.chloride IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS chloride_first
#     , FIRST_VALUE(chem.creatinine) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.creatinine IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS creatinine_first
#     , FIRST_VALUE(chem.glucose) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.glucose IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS glucose_first
#     , FIRST_VALUE(chem.sodium) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.sodium IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS sodium_first
#     , FIRST_VALUE(chem.potassium) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.potassium IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS potassium_first
#     , FIRST_VALUE(chem.albumin) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN chem.albumin IS NOT NULL THEN 0 ELSE 1 END ASC, chem.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS albumin_first

# -- From table: blood_differential
#     , FIRST_VALUE(bd.neutrophils) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN bd.neutrophils IS NOT NULL THEN 0 ELSE 1 END ASC, bd.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS neutrophils_first
#     , FIRST_VALUE(bd.lymphocytes) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN bd.lymphocytes IS NOT NULL THEN 0 ELSE 1 END ASC, bd.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS lymphocytes_first
#     , FIRST_VALUE(bd.basophils) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN bd.basophils IS NOT NULL THEN 0 ELSE 1 END ASC, bd.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS basophils_first
#     , FIRST_VALUE(bd.eosinophils) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN bd.eosinophils IS NOT NULL THEN 0 ELSE 1 END ASC, bd.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS eosinophils_first
#     , FIRST_VALUE(bd.monocytes) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN bd.monocytes IS NOT NULL THEN 0 ELSE 1 END ASC, bd.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS monocytes_first
#     , FIRST_VALUE(bd.bands) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN bd.bands IS NOT NULL THEN 0 ELSE 1 END ASC, bd.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS bands_first

# -- From table: coagulation
#     , FIRST_VALUE(coag.inr) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN coag.inr IS NOT NULL THEN 0 ELSE 1 END ASC, coag.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS inr_first
#     , FIRST_VALUE(coag.ptt) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN coag.ptt IS NOT NULL THEN 0 ELSE 1 END ASC, coag.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS ptt_first

# -- From table: enzyme
#     , FIRST_VALUE(enz.ast) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN enz.ast IS NOT NULL THEN 0 ELSE 1 END ASC, enz.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS ast_first
#     , FIRST_VALUE(enz.alt) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN enz.alt IS NOT NULL THEN 0 ELSE 1 END ASC, enz.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS alt_first
#     , FIRST_VALUE(enz.alp) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN enz.alp IS NOT NULL THEN 0 ELSE 1 END ASC, enz.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS alp_first
#     , FIRST_VALUE(enz.ld_ldh) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN enz.ld_ldh IS NOT NULL THEN 0 ELSE 1 END ASC, enz.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS ld_ldh_first
#     , FIRST_VALUE(enz.bilirubin_total) OVER
#         (PARTITION BY det.stay_id ORDER BY
#             CASE WHEN enz.bilirubin_total IS NOT NULL THEN 0 ELSE 1 END ASC, enz.charttime
#          ROWS BETWEEN unbounded preceding AND unbounded following) AS bilirubin_total_first
# FROM mimiciv_derived.icustay_detail det
# LEFT JOIN mimiciv_derived.complete_blood_count cbc
#     ON det.subject_id = cbc.subject_id
#         AND cbc.charttime >= det.admittime + '-6:00:00'
#         AND cbc.charttime <= det.admittime + '36:00:00'
# LEFT JOIN mimiciv_derived.chemistry chem
#     ON det.subject_id = chem.subject_id
#         AND chem.charttime >= det.admittime + '-6:00:00'
#         AND chem.charttime <= det.admittime + '36:00:00'
# LEFT JOIN mimiciv_derived.blood_differential bd
#     ON det.subject_id = bd.subject_id
#         AND bd.charttime >= det.admittime + '-6:00:00'
#         AND bd.charttime <= det.admittime + '36:00:00'
# LEFT JOIN mimiciv_derived.coagulation coag
#     ON det.subject_id = coag.subject_id
#         AND coag.charttime >= det.admittime + '-6:00:00'
#         AND coag.charttime <= det.admittime + '36:00:00'
# LEFT JOIN mimiciv_derived.enzyme enz
#     ON det.subject_id = enz.subject_id
#         AND enz.charttime >= det.admittime + '-6:00:00'
#         AND enz.charttime <= det.admittime + '36:00:00'  
# WHERE det.stay_id IN {tuple(eligible_stayids)}
# ORDER BY det.stay_id
# """
# lab_first_df = pd.read_sql(lab_first_query, con)
# lab_first_df

In [ ]:
# Sanity check:

# temp_q = f"""
# SELECT det.stay_id
#     , cbc.charttime
#     , wbc
#     , hematocrit
#     , hemoglobin
#     , platelet
#     , mch
#     , mchc
#     , mcv
#     , rbc
#     , rdw
# FROM mimiciv_derived.icustay_detail det
# LEFT JOIN mimiciv_derived.complete_blood_count cbc
#     ON det.subject_id = cbc.subject_id
#         AND cbc.charttime >= det.admittime + '-6:00:00'
#         AND cbc.charttime <= det.admittime + '48:00:00'
# WHERE det.stay_id IN {tuple(eligible_stayids)}
# ORDER BY det.stay_id, charttime
# """
# temp = pd.read_sql(temp_q, con)
# temp.head(500)

#### Alternative: use the last lab value measurements before receiving HFNC

In [ ]:
%%time
lab_last_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id

-- From table: complete_blood_count
    , LAST_VALUE(cbc.wbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.wbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS wbc_last
    , LAST_VALUE(cbc.hematocrit) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hematocrit IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hematocrit_last
    , LAST_VALUE(cbc.hemoglobin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.hemoglobin IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS hemoglobin_last
    , LAST_VALUE(cbc.platelet) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.platelet IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS platelet_last
    , LAST_VALUE(cbc.mch) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mch IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mch_last
    , LAST_VALUE(cbc.mchc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mchc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mchc_last
    , LAST_VALUE(cbc.mcv) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.mcv IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS mcv_last
    , LAST_VALUE(cbc.rbc) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rbc IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rbc_last
    , LAST_VALUE(cbc.rdw) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN cbc.rdw IS NOT NULL THEN 1 ELSE 0 END ASC, cbc.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS rdw_last

-- From table: chemistry
    , LAST_VALUE(chem.aniongap) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.aniongap IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS aniongap_last
    , LAST_VALUE(chem.bicarbonate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bicarbonate IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bicarbonate_last
    , LAST_VALUE(chem.bun) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.bun IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bun_last
    , LAST_VALUE(chem.calcium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.calcium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS calcium_last
    , LAST_VALUE(chem.chloride) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.chloride IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS chloride_last
    , LAST_VALUE(chem.creatinine) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.creatinine IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS creatinine_last
    , LAST_VALUE(chem.glucose) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.glucose IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS glucose_last
    , LAST_VALUE(chem.sodium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.sodium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS sodium_last
    , LAST_VALUE(chem.potassium) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.potassium IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS potassium_last
    , LAST_VALUE(chem.albumin) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN chem.albumin IS NOT NULL THEN 1 ELSE 0 END ASC, chem.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS albumin_last

-- From table: blood_differential
    , LAST_VALUE(bd.neutrophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.neutrophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS neutrophils_last
    , LAST_VALUE(bd.lymphocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.lymphocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lymphocytes_last
    , LAST_VALUE(bd.basophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.basophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS basophils_last
    , LAST_VALUE(bd.eosinophils) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.eosinophils IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS eosinophils_last
    , LAST_VALUE(bd.monocytes) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.monocytes IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS monocytes_last
    , LAST_VALUE(bd.bands) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN bd.bands IS NOT NULL THEN 1 ELSE 0 END ASC, bd.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bands_last

-- From table: coagulation
    , LAST_VALUE(coag.inr) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.inr IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS inr_last
    , LAST_VALUE(coag.ptt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN coag.ptt IS NOT NULL THEN 1 ELSE 0 END ASC, coag.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ptt_last

-- From table: enzyme
    , LAST_VALUE(enz.ast) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ast IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ast_last
    , LAST_VALUE(enz.alt) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alt IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alt_last
    , LAST_VALUE(enz.alp) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.alp IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS alp_last
    , LAST_VALUE(enz.ld_ldh) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.ld_ldh IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ld_ldh_last
    , LAST_VALUE(enz.bilirubin_total) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN enz.bilirubin_total IS NOT NULL THEN 1 ELSE 0 END ASC, enz.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS bilirubin_total_last

FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.complete_blood_count cbc
    ON det.subject_id = cbc.subject_id
        AND cbc.charttime >= st.final_starttime + '-22:00:00'
        AND cbc.charttime <= st.final_starttime + '2:00:00'
LEFT JOIN mimiciv_derived.chemistry chem
    ON det.subject_id = chem.subject_id
        AND chem.charttime >= st.final_starttime + '-22:00:00'
        AND chem.charttime <= st.final_starttime + '2:00:00'
LEFT JOIN mimiciv_derived.blood_differential bd
    ON det.subject_id = bd.subject_id
        AND bd.charttime >= st.final_starttime + '-22:00:00'
        AND bd.charttime <= st.final_starttime + '2:00:00'
LEFT JOIN mimiciv_derived.coagulation coag
    ON det.subject_id = coag.subject_id
        AND coag.charttime >= st.final_starttime + '-22:00:00'
        AND coag.charttime <= st.final_starttime + '2:00:00'
LEFT JOIN mimiciv_derived.enzyme enz
    ON det.subject_id = enz.subject_id
        AND enz.charttime >= st.final_starttime + '-22:00:00'
        AND enz.charttime <= st.final_starttime + '2:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
lab_last_df = pd.read_sql(lab_last_query, con)
lab_last_df

### 3.5 Blood gas

In [ ]:
%%time
bg_query = f"""
SELECT det.stay_id
, lactate_min, lactate_max
, ph_min, ph_max
, po2_min, po2_max
, pco2_min, pco2_max
, baseexcess_min, baseexcess_max
, totalco2_min, totalco2_max
FROM mimiciv_derived.icustay_detail det
LEFT JOIN mimiciv_derived.first_day_bg_art fdbg ON fdbg.stay_id = det.stay_id
WHERE det.stay_id IN {tuple(eligible_stayids)}
"""
bg_df = pd.read_sql(bg_query, con)
bg_df

#### Alternative: use the last measurements before receiving HFNC

In [ ]:
%%time
bg_last_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
    , LAST_VALUE(lactate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN lactate IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS lactate_last
    , LAST_VALUE(ph) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN ph IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS ph_last
    , LAST_VALUE(so2)OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN so2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS so2_last
    , LAST_VALUE(po2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN po2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS po2_last
    , LAST_VALUE(pco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN pco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS pco2_last
    , LAST_VALUE(baseexcess) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN baseexcess IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS baseexcess_last
    , LAST_VALUE(totalco2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN totalco2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS totalco2_last
    , LAST_VALUE(fio2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN fio2 IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS fio2_other_last
    , LAST_VALUE(fio2_chartevents) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN fio2_chartevents IS NOT NULL THEN 1 ELSE 0 END ASC, bg.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS fio2_chartevents_last

FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.bg bg
    ON det.subject_id = bg.subject_id
        AND bg.charttime >= st.final_starttime + '-22:00:00'
        AND bg.charttime <= st.final_starttime + '2:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
;
"""
bg_last_df = pd.read_sql(bg_last_query, con)

bg_last_df['fio2_obs_last'] = np.where(
    ~bg_last_df.fio2_chartevents_last.isnull(),
    bg_last_df.fio2_chartevents_last,
    bg_last_df.fio2_other_last
)
bg_last_df

### 3.6 HFNO Settings (Flow Rate)

In [ ]:
# settings_query = f"""
# {hfnc_subquery}

# , flow_rates AS (
#     SELECT ce.stay_id
#     , ce.charttime
#     , ce.itemid
#     , final_starttime
#     , final_endtime
#     , ce.valuenum AS flow_rate_num
#     FROM mimiciv_icu.chartevents ce
#     LEFT JOIN support_table st
#     ON st.stay_id = ce.stay_id
#         AND ce.charttime >= st.final_starttime 
#         AND ce.charttime <= st.final_endtime
#     WHERE ce.itemid IN
#         (
#             --224691, -- Flow Rate (L)
#             227287, -- additional o2 flow
#             223834  -- o2 flow
#         )
#         AND ce.stay_id IN {tuple(eligible_stayids)}
#         AND ce.charttime >= st.final_starttime
#         AND ce.charttime <= st.final_endtime
#     ORDER BY ce.stay_id
# )

# SELECT stay_id
# , MIN(flow_rate_num) AS min_flow_rate
# , AVG(flow_rate_num) AS mean_flow_rate
# , MAX(flow_rate_num) AS max_flow_rate
# FROM flow_rates
# GROUP BY stay_id
# ORDER BY stay_id



# """
# settings_df = pd.read_sql(settings_query, con)
# settings_df

settings_query = f"""
{hfnc_subquery}

, flow_rates AS (
    SELECT
        ce.stay_id
      , ce.charttime
      , ce.itemid
      , st.final_starttime
      , st.final_endtime
      , ce.valuenum AS flow_rate_num
    FROM mimiciv_icu.chartevents ce
    JOIN support_table st
      ON st.stay_id = ce.stay_id
     AND ce.charttime >= st.final_starttime
     AND ce.charttime <= st.final_endtime
    WHERE ce.itemid IN (227287, 223834)
      AND ce.stay_id IN {tuple(eligible_stayids)}
)

SELECT
    det.stay_id
  , fr_pick.flow_rate_num AS flow_rate_last
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
  ON det.stay_id = st.stay_id
LEFT JOIN (
    SELECT DISTINCT ON (fr.stay_id)
        fr.stay_id
      , fr.flow_rate_num
    FROM flow_rates fr
    JOIN support_table st2
      ON st2.stay_id = fr.stay_id
    WHERE fr.charttime >= st2.final_starttime - interval '22 hour'
      AND fr.charttime <= st2.final_starttime + interval '2 hour'
    ORDER BY
        fr.stay_id
      , (fr.flow_rate_num IS NOT NULL) DESC
      , fr.charttime DESC
) fr_pick
  ON fr_pick.stay_id = det.stay_id
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
settings_df = pd.read_sql(settings_query, con)
settings_df


In [ ]:
# settings_df.mean_flow_rate.hist(alpha=0.5, label='mean')
# settings_df.max_flow_rate.hist(alpha=0.5, label='max')
# plt.legend()

In [ ]:
# Extract last fio2 for ROX calculation
last_fio2_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
, LAST_VALUE(fio2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN fio2 IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS fio2_setting_last
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.ventilator_setting vs
    ON det.stay_id = vs.stay_id
        AND vs.charttime >= st.final_starttime + '-22:00:00'
        AND vs.charttime <= st.final_starttime + '2:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
last_fio2_df = pd.read_sql(last_fio2_query, con)
last_fio2_df

In [ ]:
# Extract last respiratory_rate for ROX calculation
last_vitals_query = f"""
{hfnc_subquery}

SELECT DISTINCT det.stay_id
, LAST_VALUE(resp_rate) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN resp_rate IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS resp_rate_last
, LAST_VALUE(spo2) OVER
        (PARTITION BY det.stay_id ORDER BY
            CASE WHEN spo2 IS NOT NULL THEN 1 ELSE 0 END ASC, vs.charttime
         ROWS BETWEEN unbounded preceding AND unbounded following) AS spo2_last
FROM mimiciv_derived.icustay_detail det
LEFT JOIN support_table st
    ON det.stay_id = st.stay_id
LEFT JOIN mimiciv_derived.vitalsign vs
    ON det.stay_id = vs.stay_id
        AND vs.charttime >= st.final_starttime + '-22:00:00'
        AND vs.charttime <= st.final_starttime + '2:00:00'
WHERE det.stay_id IN {tuple(eligible_stayids)}
ORDER BY det.stay_id
"""
last_vitals_df = pd.read_sql(last_vitals_query, con)
last_vitals_df


## 4. Combine tables

In [ ]:
# df = pd.concat([demo_df, comorb_df, vital_df, lab_first_df, bg_first_df], axis=1, verify_integrity=True)

df = demo_df.set_index('stay_id').join(comorb_df.set_index('stay_id'), how='left')
df = df.join(mean_vital_df.set_index('stay_id'), how='left')
# df = df.join(lab_first_df.set_index('stay_id'), how='left')
# df = df.join(bg_first_df.set_index('stay_id'), how='left')
df = df.join(bg_last_df.set_index('stay_id'), how='left')
df = df.join(lab_last_df.set_index('stay_id'), how='left')
# df = df.join(vs_last_df.set_index('stay_id'), how='left')
df = df.join(vs_mean_last24h_df.set_index('stay_id'), how='left')
# df = df.join(uo_last24h_df.set_index('stay_id'), how='left')
df = df.join(fluidbalance_df.set_index('stay_id'), how='left')
df = df.join(vaso_df.set_index('stay_id'), how='left')
df = df.join(settings_df.set_index('stay_id'), how='left')
df = df.join(last_vitals_df.set_index('stay_id'), how='left')
df = df.join(last_fio2_df.set_index('stay_id'), how='left')

print(df.columns)
df

In [ ]:
hfno_df

In [ ]:
# Add intubation and death flags:
# Fix: use index-based alignment (hfno_df is ORDER BY stay_id; df index is stay_id)
# Using .values would assign outcomes positionally to wrong patients if row orders differ.
hfno_indexed = hfno_df.set_index('stay_id')

outcome_cols = [
    'duration', 'deathtime', 'iv_time', 'final_starttime', 'final_endtime',
    'last_vent_type', 'intubated', 'died', 'inhosp_mortality', 'admittime',
]
for col in outcome_cols:
    if col in hfno_indexed.columns:
        df[col] = hfno_indexed[col]

df['fio2_last'] = np.where(
    ~df.fio2_setting_last.isnull(),
    df.fio2_setting_last,
    df.fio2_obs_last
)
df

In [ ]:
from pathlib import Path

# Create folder if it doesn't exist
Path("filtered_data").mkdir(parents=True, exist_ok=True)
Path("processed_data").mkdir(parents=True, exist_ok=True)
Path("results").mkdir(parents=True, exist_ok=True)


In [ ]:
df.to_csv('./processed_data/mimiciv_hfno_data.csv')